In [5]:
import tifffile
import os
import pandas as pd

input_directory = r"E:\ExpEM011_Imaging_20x_2ndtry"
for dir in os.listdir(input_directory):
    if os.path.isdir(os.path.join(input_directory, dir)):
        proprties_folder = os.path.join(input_directory, dir, "properties")
        if os.path.exists(proprties_folder):
            for file in os.listdir(proprties_folder):
                if file.endswith(".csv"):
                    # delte the csv file
                    os.remove(os.path.join(proprties_folder, file))

In [5]:
import tifffile

import os
import re
import json
import numpy as np


def file_to_folder(input_directory):
    import os
    import shutil

    for file in os.listdir(input_directory):
        file_path = os.path.join(input_directory, file)

        if os.path.isfile(file_path) and file.endswith(
            (".tif", ".tiff", ".ims", ".nd")
        ):
            folder_name = os.path.splitext(file)[0]
            new_folder_path = os.path.join(input_directory, folder_name)
            os.makedirs(new_folder_path, exist_ok=True)

            if not file_path.endswith(".nd"):
                shutil.move(file_path, os.path.join(new_folder_path, file))
            else:
                shutil.move(file_path, os.path.join(new_folder_path, file))
                base_name = os.path.splitext(file)[0]
                other_files = [
                    f
                    for f in os.listdir(input_directory)
                    if f.startswith(base_name) and f.lower().endswith(".stk")
                ]
                for other_file in other_files:
                    shutil.move(
                        os.path.join(input_directory, other_file),
                        os.path.join(new_folder_path, other_file),
                    )
                output_tif = nd_to_tif(os.path.join(new_folder_path, file))
                if os.path.exists(output_tif):
                    os.remove(os.path.join(new_folder_path, file))
                    for other_file in other_files:
                        os.remove(
                            os.path.join(new_folder_path, other_file)
                        )

    return


def _natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]


def _parse_nd_metadata(nd_path):
    meta = {}
    with open(nd_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith(";"):
                continue
            m = re.match(r'"?([^",]+)"?\s*,\s*"?(.*?)"?$', line)
            if m:
                key = m.group(1).strip()
                val = m.group(2).strip()
                meta[key] = val
    return meta


def _to_int(value):
    try:
        return int(float(value))
    except Exception:
        return None


def _to_float(value):
    try:
        return float(value)
    except Exception:
        return None


def _get_first_numeric(meta, keys):
    for k in keys:
        if k in meta:
            v = _to_float(meta[k])
            if v is not None:
                return v
    return None


def nd_to_tif(input_file):
    """Convert MetaMorph ND + STK series to TIFF with OME metadata, axes T,Z,C,Y,X."""
    nd_meta = _parse_nd_metadata(input_file)
    input_dir = os.path.dirname(input_file)
    base_name = os.path.splitext(os.path.basename(input_file))[0]

    stk_files = sorted(
        [
            f
            for f in os.listdir(input_dir)
            if f.startswith(base_name) and f.lower().endswith(".stk")
        ],
        key=_natural_key,
    )
    if not stk_files:
        raise FileNotFoundError(f"No STK files found for {input_file}")

    records = []  # (t_idx, c_idx, stack, filename)
    for f in stk_files:
        full = os.path.join(input_dir, f)
        stack = tifffile.imread(full)
        if stack.ndim == 2:
            stack = stack[np.newaxis, :, :]  # Z, Y, X
        elif stack.ndim != 3:
            raise ValueError(f"Unexpected STK shape {stack.shape} for {f}")

        m_t = re.search(r"(?:^|[_-])t(\d+)", f, flags=re.IGNORECASE)
        m_w = re.search(r"(?:^|[_-])w(\d+)", f, flags=re.IGNORECASE)
        t_token = int(m_t.group(1)) if m_t else None
        c_token = int(m_w.group(1)) if m_w else None
        records.append((t_token, c_token, stack, f))

    first_shape = records[0][2].shape
    for _, _, st, fn in records[1:]:
        if st.shape != first_shape:
            raise ValueError(
                f"STK stacks are not the same shape: {records[0][3]} {first_shape} vs {fn} {st.shape}"
            )

    z, y, x = first_shape
    token_ts = [r[0] for r in records if r[0] is not None]
    token_cs = [r[1] for r in records if r[1] is not None]

    n_t_meta = _to_int(nd_meta.get("NTimePoints") or nd_meta.get("NumberOfTimePoints"))
    n_c_meta = _to_int(nd_meta.get("NWavelengths") or nd_meta.get("NumberOfWavelengths"))

    n_t = max(token_ts) if token_ts else (n_t_meta if n_t_meta else None)
    n_c = max(token_cs) if token_cs else (n_c_meta if n_c_meta else None)

    if n_t is None and n_c is None:
        n_t, n_c = len(records), 1
    elif n_t is None:
        n_t = int(np.ceil(len(records) / n_c))
    elif n_c is None:
        n_c = int(np.ceil(len(records) / n_t))

    one_based_t = len(token_ts) > 0 and min(token_ts) >= 1
    one_based_c = len(token_cs) > 0 and min(token_cs) >= 1

    # Final output order: T, Z, C, Y, X
    data = np.zeros((n_t, z, n_c, y, x), dtype=records[0][2].dtype)

    for i, (t_token, c_token, stack, _) in enumerate(records):
        if t_token is None:
            t_idx = i // n_c
        else:
            t_idx = (t_token - 1) if one_based_t else t_token

        if c_token is None:
            c_idx = i % n_c
        else:
            c_idx = (c_token - 1) if one_based_c else c_token

        if t_idx >= n_t or c_idx >= n_c or t_idx < 0 or c_idx < 0:
            raise IndexError(
                f"Computed index out of bounds for record {i}: t={t_idx}, c={c_idx}, shape={(n_t, z, n_c, y, x)}"
            )
        data[t_idx, :, c_idx, :, :] = stack

    metadata = {"axes": "TZCYX"}

    ch_names = []
    for idx in range(1, n_c + 1):
        key = f"WaveName{idx}"
        if key in nd_meta and nd_meta[key]:
            ch_names.append(nd_meta[key])
    if len(ch_names) == n_c:
        metadata["Channel"] = {"Name": ch_names}

    px = _get_first_numeric(nd_meta, ["PixelSizeX", "XCalibrationMicrons", "XCalibration", "dXCalibration"])
    py = _get_first_numeric(nd_meta, ["PixelSizeY", "YCalibrationMicrons", "YCalibration", "dYCalibration"])
    pz = _get_first_numeric(nd_meta, ["ZStep", "ZStepSize", "StepSize", "dZStep"])
    dt = _get_first_numeric(nd_meta, ["TimeIncrement", "SamplingInterval", "Interval", "dTimeStep"])

    if px and px > 0:
        metadata["PhysicalSizeX"] = px
        metadata["PhysicalSizeXUnit"] = "um"
    if py and py > 0:
        metadata["PhysicalSizeY"] = py
        metadata["PhysicalSizeYUnit"] = "um"
    if pz and pz > 0:
        metadata["PhysicalSizeZ"] = pz
        metadata["PhysicalSizeZUnit"] = "um"
    if dt and dt > 0:
        metadata["TimeIncrement"] = dt
        metadata["TimeIncrementUnit"] = "s"

    output_file = os.path.splitext(input_file)[0] + ".tif"
    tifffile.imwrite(
        output_file,
        data,
        ome=True,
        metadata=metadata,
        bigtiff=True,
        compression="zlib",
        compressionargs={"level": 1},
    )

    sidecar = os.path.splitext(output_file)[0] + "_nd_metadata.json"
    with open(sidecar, "w", encoding="utf-8") as f:
        json.dump(nd_meta, f, indent=2, ensure_ascii=False)

    return output_file


input_directory = r"C:\Users\6331823\Local SSD\Data_Mo\ND files"

file_to_folder(input_directory)

In [ ]:
input_directory = r"C:\Users\6331823\Local SSD\Data_Mo\ND files"
output_directory = r"C:\Users\6331823\Local SSD\Data_Mo\Train model mo"

for dir in os.listdir(input_directory):
    if os.path.isdir(os.path.join(input_directory, dir)):
        image = os.path.join(input_directory, dir, dir + ".tif")
        image = tifffile.imread(image)
        nuclei = image[0]
        random_index = np.random.randint(0, nuclei.shape[0], size=10)
        for i, idx in enumerate(random_index):
            tifffile.imwrite(
                os.path.join(output_directory, f"{dir}_nuclei_{i}.tif"),
                nuclei[idx],
            )

In [11]:
import numpy as np
from pathlib import Path

src = Path(r"C:\Users\6331823\Local SSD\Data_Mo\Train model mo\extra_3_seg.npy")
dst = src.with_name(src.stem + "_neg_seg.npy")

obj = np.load(src, allow_pickle=True)

# Handle common .npy layouts: pickled dict, object scalar, object array, or [image, mask] array
if isinstance(obj, np.ndarray) and obj.dtype == object and obj.shape == ():
    obj = obj.item()

if isinstance(obj, dict):
    # Most robust: zero any key that looks like a mask
    new_obj = dict(obj)
    mask_keys = [k for k in new_obj.keys() if "mask" in str(k).lower()]
    if not mask_keys:
        raise ValueError(f"No mask-like key found in dict. Keys: {list(new_obj.keys())}")
    for k in mask_keys:
        new_obj[k] = np.zeros_like(new_obj[k])
    np.save(dst, new_obj, allow_pickle=True)

elif isinstance(obj, np.ndarray) and obj.dtype == object and obj.size >= 2:
    seq = list(obj)
    seq[1] = np.zeros_like(seq[1])
    np.save(dst, np.array(seq, dtype=object), allow_pickle=True)

elif isinstance(obj, (list, tuple)) and len(obj) >= 2:
    seq = list(obj)
    seq[1] = np.zeros_like(seq[1])
    np.save(dst, np.array(seq, dtype=object), allow_pickle=True)

else:
    raise ValueError(f"Unsupported structure: type={type(obj)}, dtype={getattr(obj, 'dtype', None)}, shape={getattr(obj, 'shape', None)}")

print(f"Saved: {dst}")
print(f"Exists: {dst.exists()}")

Saved: C:\Users\6331823\Local SSD\Data_Mo\Train model mo\extra_3_seg_neg_seg.npy
Exists: True


In [ ]:
import napari
import tifffile
import os

input_directory = r"C:\Users\6331823\Local SSD\Data_Mo\ND files\20251216_gapstrain_1_1"
segmentation_cleaned = tifffile.imread(
    os.path.join(input_directory, f"segmentation_cleaned.tif")
)
segmentation_old = tifffile.imread(
    os.path.join(input_directory, f"{os.path.basename(input_directory)}_segmented old.tif")
)
segmentation_new = tifffile.imread(
    os.path.join(input_directory, f"{os.path.basename(input_directory)}_segmented.tif")
)
image = tifffile.imread(
    os.path.join(input_directory, f"20251216_gapstrain_1_1_cropped.tif")
)[:,0]
scale = (0.25, 0.057, 0.057)  # Example scale values for Z, Y, X in microns
viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(segmentation_cleaned, name="segmentation_cleaned", scale = scale)
viewer.add_labels(segmentation_old, name="segmentation_old", scale = scale)
viewer.add_labels(segmentation_new, name="segmentation_new", scale = scale)
viewer.add_image(image, name="image", scale = scale)

<Image layer 'image' at 0x2b05267ea70>

c:\Users\6331823\AppData\Local\anaconda3\envs\nuclei_segmenter\lib\site-packages\napari\layers\labels\labels.py:908: UserWarning: Contours are not displayed during 3D rendering
  warnings.warn(
c:\Users\6331823\AppData\Local\anaconda3\envs\nuclei_segmenter\lib\site-packages\napari\layers\labels\labels.py:908: UserWarning: Contours are not displayed during 3D rendering
  warnings.warn(


In [9]:
for z in range(segmentated.shape[0]):
    tifffile.imwrite(
        os.path.join(r"C:\Users\6331823\Local SSD\Data_Mo\Train model mo tiff", f"1_1_z{z}_masks.tif"),
        segmentated[z],
    )
    tifffile.imwrite(
        os.path.join(r"C:\Users\6331823\Local SSD\Data_Mo\Train model mo tiff", f"1_1_z{z}_img.tif"),
        image[z],
    )

In [10]:
from cellpose import io, models, train

train_dir = r"C:\Users\6331823\Local SSD\Data_Mo\Train model mo tiff"
test_dir = r"C:\Users\6331823\Local SSD\Data_Mo\Train model mo tiff"
io.logger_setup()

output = io.load_train_test_data(
    train_dir,
    test_dir,
    image_filter="_img",
    mask_filter="_masks",
    look_one_level_down=False,
)
images, labels, image_names, test_images, test_labels, image_names_test = output

model = models.CellposeModel(gpu=True)

model_path, train_losses, test_losses = train.train_seg(
    model.net,
    train_data=images,
    train_labels=labels,
    test_data=test_images,
    test_labels=test_labels,
    weight_decay=0.1,
    learning_rate=1e-5,
    n_epochs=100,
    model_name="mo_tiff",
    min_train_masks=1,
)

creating new log file
2026-03-18 17:21:24,333 [INFO] WRITING LOG OUTPUT TO C:\Users\6331823\.cellpose\run.log
2026-03-18 17:21:24,333 [INFO] 
cellpose version: 	4.0.8 
platform:       	win32 
python version: 	3.10.19 
torch version:  	2.10.0+cu128
2026-03-18 17:21:24,404 [INFO] not all flows are present, running flow generation for all images
2026-03-18 17:21:25,185 [INFO] 61 / 61 images in C:\Users\6331823\Local SSD\Data_Mo\Train model mo tiff folder have labels
2026-03-18 17:21:25,229 [INFO] not all flows are present, running flow generation for all images
2026-03-18 17:21:25,393 [INFO] 61 / 61 images in C:\Users\6331823\Local SSD\Data_Mo\Train model mo tiff folder have labels
2026-03-18 17:21:25,559 [INFO] ** TORCH CUDA version installed and working. **
2026-03-18 17:21:25,559 [INFO] >>>> using GPU (CUDA)
2026-03-18 17:21:26,951 [INFO] >>>> loading model C:\Users\6331823\.cellpose\models\cpsam
2026-03-18 17:21:27,851 [INFO] computing flows for labels


100%|██████████| 61/61 [00:03<00:00, 18.35it/s]

2026-03-18 17:21:31,239 [INFO] computing flows for labels



100%|██████████| 61/61 [00:03<00:00, 20.18it/s]

2026-03-18 17:21:34,317 [INFO] >>> computing diameters



  0%|          | 0/61 [00:00<?, ?it/s]c:\Users\6331823\AppData\Local\anaconda3\envs\nuclei_segmenter\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\6331823\AppData\Local\anaconda3\envs\nuclei_segmenter\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 61/61 [00:00<00:00, 431.50it/s]

2026-03-18 17:21:34,650 [WARNING] 31 train images with number of masks less than min_train_masks (5), removing from train set
2026-03-18 17:21:34,664 [INFO] >>> normalizing {'lowhigh': None, 'percentile': None, 'normalize': True, 'norm3D': True, 'sharpen_radius': 0, 'smooth_radius': 0, 'tile_norm_blocksize': 0, 'tile_norm_smooth3D': 1, 'invert': False}


2026-03-18 17:21:35,762 [INFO] >>> n_epochs=100, n_train=30, n_test=61
2026-03-18 17:21:35,763 [INFO] >>> AdamW, learning_rate=0.00001, weight_decay=0.10000
2026-03-18 17:21:35,765 [INFO] >>> saving model to c:\Users\6331823\Local SSD\Coding projects\organoid_segmenter\miscellaneous\models\mo_tiff
2026-03-18 17:21:43,011 [INFO] 0, train_loss=0.1212, test_loss=0.0825, LR=0.000000, time 7.25s
2026-03-18 17:22:04,113 [INFO] 5, train_loss=0.0891, test_loss=0.0741, LR=0.000006, time 28.35s
2026-03-18 17:22:24,205 [INFO] 10, train_loss=0.0928, test_loss=0.0501, LR=0.000010, time 48.44s
2026-03-18 17:23:03,087 [INFO] 20, train_loss=0.0634, test_loss=0.0433, LR=0.000010, time 87.32s
2026-03-18 17:23:42,222 [INFO] 30, train_loss=0.0731, test_loss=0.0394, LR=0.000010, time 126.46s
2026-03-18 17:24:20,197 [INFO] 40, train_loss=0.0583, test_loss=0.0382, LR=0.000010, time 164.43s
2026-03-18 17:24:58,265 [INFO] 50, train_loss=0.0477, test_loss=0.0377, LR=0.000005, time 202.50s
2026-03-18 17:25:37,71